# Tutoriel -- pipeline "critical materials"

Ce notebook explique :
1. Comment fonctionnent les 3 pipelines qui génèrent les fichiers `.dat` (à partir des fichiers Excel)
2. Comment utiliser `run_pathway_materials()` -- tous les paramètres qu'on a définis, et ce qu'ils font
3. Comment lire les résultats et régénérer le dashboard

Tout se lance depuis `projects/critical_materials/` (le dossier de ce notebook).

In [5]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, 'src')

from run_pathway_materials import run_pathway_materials
from Plot_functions import build_materials_dashboard, build_scenario_selector
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Les pipelines -- comment sont générés les `.dat`

Rien n'est écrit à la main dans `ampl_files/*.dat` : ce sont des fichiers **auto-générés** par des scripts Python
qui lisent les fichiers Excel dans `excel_files/`. Si tu modifies un Excel, il faut **relancer le pipeline
correspondant** pour que le changement se propage dans le `.dat`, puis relancer `run_pathway_materials`.

| Pipeline | Source Excel | Sortie | Script |
|---|---|---|---|
| `mi_pipeline` | `Material_intensities_energyscope.xlsx` | `Material_intensity.dat` (+ `technologies_mi_all_years.xlsx`, audit) | `run_build_mi.py` |
| `rr_pipeline` | `Recycling_rates.xlsx` | `Material_recycling.dat` | `run_build_rr.py` |
| `rt_pipeline` | `Recycling_rates.xlsx` | `Material_recycling_process.dat` | `run_build_rt.py` |

`Material_intensity.dat` et `Material_recycling.dat` (les 2 premiers) sont ceux qu'on utilise activement
(approche `recycling_materials`). Le 3e (`rt_pipeline`, approche `recycling_materials_technologies` --
technologies de recyclage en compétition, PV c-Si) **est mis de côté pour l'instant** -- ne pas utiliser
`materials_recycling_process=True` tant qu'on n'y est pas revenu, `Constraints_recycling_technologies.mod`
n'est plus synchronisé avec `Constraints.mod` (Constraints.mod a été remis 100% pur Approche 1).

### 1.1 `mi_pipeline` -- intensités matérielles (`material_intensity`)

Lit `Mapping` (quelle techno EnergyScope correspond à quelle sous-techno de la littérature) +
`MI_Energy`/`MI_Vehicles`/`MI_Vehicles_Bieuville_Clean`/`MI_Vehicles_Public`/`MI_H2` (les valeurs elles-mêmes).

**Astuce vitesse** : générer `technologies_mi_all_years.xlsx` (le fichier d'audit, avec le détail
`Vehicle_Calc_Detail` de chaque conversion g/véhicule -> t/(pkm/h)) est **beaucoup plus lent** que générer
juste le `.dat` (~5 min contre quelques secondes, car il réécrit ~200k lignes avec mise en forme). Si tu
n'as pas besoin de l'audit (juste besoin du `.dat` à jour pour lancer un run), passe `write_xlsx=False`.

In [ ]:
from run_build_mi import main as build_mi

# Génère Material_intensity.dat + technologies_mi_all_years.xlsx (lent, ~5 min)
build_mi()

# Rapide : juste le .dat, sans régénérer l'xlsx d'audit
# build_mi(write_xlsx=False)

# Autres options :
# build_mi(vehicle_source='bieuville')   # source alternative pour les intensités véhicules
# build_mi(scenario='optimiste')          # applique les overrides du scénario 'optimiste' (feuille Overrides)

### 1.2 `rr_pipeline` -- taux de recyclage (`recycling_rate`), approche `recycling_materials`

Lit `Mapping` + `RR_Energy`/`RR_Vehicles`/`RR_Vehicles_Public`/`RR_H2` (taux spécifiques par techno) +
`RR_Global` (taux global par matériau, littérature Graedel et al. 2022 -- 1re des 3 colonnes) +
`Recycling_objective` (cible pour `follow_objective=True`, voir plus bas).

**Logique cellule par cellule** (important) : pour chaque (techno, matériau), le taux **spécifique**
(RR_Energy/RR_Vehicles/...) est utilisé s'il existe ; sinon on retombe sur le taux **global** de
`RR_Global` pour ce matériau. Ça s'applique aussi bien aux technos sans mapping du tout (ex: nucléaire,
hydro) qu'aux technos mappées mais incomplètes (ex: l'éolien n'a qu'1 matériau sur 41 avec un vrai taux
spécifique -- tout le reste retombe sur `RR_Global`).

In [ ]:
from run_build_rr import main as build_rr

build_rr()

# build_rr(scenario='optimiste')   # applique les overrides du scénario 'optimiste'
# build_rr(write_dat=False)         # juste le rapport de couverture, sans écrire le .dat

### 1.3 `rt_pipeline` -- technologies de recyclage en compétition, approche `recycling_materials_technologies`

**Mise de côté pour l'instant** (voir note plus haut) -- `python run_build_rt.py` régénère quand même
`Material_recycling_process.dat` correctement, mais `materials_recycling_process=True` dans
`run_pathway_materials` va planter tant que `Constraints_recycling_technologies.mod` n'a pas été
resynchronisé avec `Constraints.mod`. À reprendre plus tard.

## 2. `run_pathway_materials()` -- tous les paramètres

Lance une résolution du modèle pathway + contraintes matières, et retourne un dict de résultats
(DataFrames pandas). Sauvegarde aussi les résultats dans `out/<case_study>/` et génère un dashboard
HTML par défaut.

| Paramètre | Défaut | Rôle |
|---|---|---|
| `case_study` | *(obligatoire)* | Nom du run -- sert de nom de dossier `out/<case_study>/` |
| `N_year_opti` | `30` | Durée de la fenêtre glissante d'optimisation [années]. 30 = tout l'horizon 2020-2050 en une seule fenêtre |
| `N_year_overlap` | `0` | Chevauchement entre fenêtres consécutives [années] |
| `gwp_budget` | `False` | Budget GWP cumulé sur toute la transition [kt CO2-eq.]. `False` = désactivé |
| `CO2_neutrality_2050` | `True` | Si `True`, force `gwp_limit['YEAR_2050'] = CO2_neutrality_2050_val` |
| `description` | `''` | Description courte, stockée dans le CSV récapitulatif |
| `save_pkl` | `True` | Sauvegarde `_Results.pkl` + `_Materials_Results.pkl` dans `out/<case_study>/` |
| `skip_if_exists` | `False` | Si `True` et que le pkl existe déjà, recharge depuis le disque au lieu de relancer le solve |
| `verbose` | `False` | Affiche les logs AMPL/Gurobi (utile pour déboguer un solve qui rame) |
| `crossover` | `0` | Option Gurobi (0 = pas de crossover après le barrier method, plus rapide) |
| `hydro_quebec_constraints` | `True` | Contraintes minimums éolien/hydro/PV du plan Hydro-Québec |
| `materials_limit` | `False` | Charge `Material_limits.dat` (plafonds manuels `limit_material`/`limit_material_year`) |
| `materials_recycling` | `False` | Charge `Material_recycling.dat` (approche `recycling_materials`) -- **sans ça, aucun recyclage** (`recycling_rate` reste à 0 partout) |
| `follow_objective` | `False` | Seulement utile si `materials_recycling=True`. `False` = l'optimiseur recycle librement jusqu'au plafond `recycling_rate` (le moins cher, vu que recycler est quasi gratuit face au coût d'enfouissement 0.01 $/t). `True` = force une **égalité exacte** avec `recycling_objective_share` (feuille `Recycling_objective`), pas juste un minimum |
| `materials_recycling_process` | `False` | Approche `recycling_materials_technologies` -- **en pause, ne pas utiliser pour l'instant** (voir §1.3) |
| `build_dashboard` | `True` | Génère `out/<case_study>/materials_graphs/` après le run |

**Toujours actif, peu importe les paramètres** : `Material_intensity.dat` (le fichier généré par
`mi_pipeline`) est systématiquement chargé -- c'est lui qui donne `material_intensity`, la quantité de
base (utilisée pour calculer `Material_content_year`, `Decommissioned_material`, etc.).

### 2.1 Exemple -- Approche 1, l'optimiseur décide librement

`follow_objective=False` : l'optimiseur recycle jusqu'au plafond technique (`recycling_rate`), pas plus,
pas moins -- pas de contrainte d'objectif imposée.

In [ ]:
results_free = run_pathway_materials(
    'recycling_materials',
    materials_recycling=True,
    follow_objective=False,
    description="Approche 1 -- optimiseur libre.",
)

rec = results_free['Recycled_material']['Recycled_material']
rec[rec.abs() > 1e-9].groupby('Materials').sum().sort_values(ascending=False)

### 2.2 Exemple -- Approche 1, égalité forcée avec le scénario

`follow_objective=True` : force `recycling_objective_share` (feuille `Recycling_objective`) exactement,
agrégé sur toutes les technos qui ont un vrai `recycling_rate` pour ce matériau (le pipeline `rr_pipeline`
clippe automatiquement la cible à ce qui est techniquement atteignable, pour éviter une infaisabilité).

In [ ]:
results_objective = run_pathway_materials(
    'recycling_materials_objective',
    materials_recycling=True,
    follow_objective=True,
    description="Approche 1 -- égalité recycling_objective_share.",
)

rec = results_objective['Recycled_material']['Recycled_material']
rec[rec.abs() > 1e-9].groupby('Materials').sum().sort_values(ascending=False)

### 2.3 Exemple -- recharger un run déjà fait (`skip_if_exists`)

Si `out/<case_study>/_Results.pkl` existe déjà, `skip_if_exists=True` recharge directement depuis le
disque au lieu de relancer tout le solve (~2-6 min économisées) -- pratique pour retravailler le
dashboard ou explorer les résultats sans re-solver.

In [ ]:
results_free = run_pathway_materials(
    'recycling_materials',
    materials_recycling=True,
    follow_objective=False,
    skip_if_exists=True,
)

## 3. Le dict de résultats

`run_pathway_materials` retourne un dict de DataFrames pandas. Les clés les plus utiles pour les
matériaux (en plus de tout ce que `run_pathway` normal retourne déjà -- `F_new`, `F_Mult`, `Assets`,
`TotalCost`, ...) :

| Clé | Description |
|---|---|
| `Material_content_year` | Demande matière annuelle [kt/an], indexé (Years, Technologies, Materials) |
| `Material_content_cumulative` | Cumul de `Material_content_year` sur l'horizon -- la dernière année = le total |
| `Decommissioned_material` | Matière démantelée mécaniquement [kt/an], avant toute décision de recyclage |
| `Recycled_material` | Matière effectivement recyclée [kt/an] |
| `Recycled_material_cumulative` | Cumul de `Recycled_material` |
| `Disposed_material` | Matière enfouie/incinérée [kt/an] = `Decommissioned_material - Recycled_material` |

Chaque DataFrame a une seule colonne (même nom que la clé) et un MultiIndex `(Years, Technologies,
Materials)`.

In [ ]:
results_free['Material_content_year'].head()

## 4. Dashboard

`build_dashboard=True` (par défaut dans `run_pathway_materials`) génère automatiquement
`out/<case_study>/materials_graphs/index.html` **et** régénère `out/index.html` (le sélecteur qui liste
tous les runs existants) -- pas besoin de le faire à la main.

Si tu veux juste régénérer le dashboard d'un run déjà fait (sans re-solver), utilise `skip_if_exists=True`
(§2.3) -- le dashboard se reconstruit à partir du pkl rechargé.

In [ ]:
# Régénère juste le sélecteur out/index.html (rarement nécessaire, c'est automatique après chaque run)
build_scenario_selector()

In [7]:
results = run_pathway_materials(
    'Test_realiste',
    verbose=True,
    materials_recycling=False,
    materials_recycling_cost=False,   # recycling_cost/primary_material_cost=0, disposal_cost=0.01
    follow_objective_full=False,       # force le recyclage à 100% du plafond atteignable, sans coût
)


Presolve eliminates 0 constraints and 209047 variables.
Adjusted problem:
1365011 variables:
	6447 binary variables
	84 nonlinear variables
	1358480 linear variables
1402423 constraints; 5705564 nonzeros
	7 nonlinear constraints
	1402416 linear constraints
	1053075 equality constraints
	345857 inequality constraints
	3491 range constraints
2 objectives, all linear; 3 nonzeros.

Gurobi 13.0.2:   pre:dual = -1
  alg:method = 2
  bar:crossover = 0
  tech:threads = 0
  pre:passes = 3
  bar:convtol = 9.9999999999999995e-07
  pre:solve = -1
  iis:find = 1
Set parameter LogToConsole to value 1
  tech:outlev = 1

AMPL MP initial flat model has 1365011 variables (0 integer, 6447 binary);
Objectives: 1 linear; 
Constraints:  1402423 linear;

AMPL MP final model has 1368502 variables (0 integer, 6447 binary);
Objectives: 1 linear; 
Constraints:  1396961 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[x86] - Darwin 23.5.0 23F79)

CPU model: I

In [ ]:
results = run_pathway_materials(
    'Test_realiste',
    verbose=True,
    materials_recycling=False,
    materials_recycling_cost=False,   # recycling_cost/primary_material_cost=0, disposal_cost=0.01
    follow_objective_full=False,       # force le recyclage à 100% du plafond atteignable, sans coût
)

In [ ]:
results = run_pathway_materials(
    'recycling_materials_no_cost',
    verbose=True,
    materials_recycling=True,
    materials_recycling_cost=False,   # recycling_cost/primary_material_cost=0, disposal_cost=0.01
    follow_objective_full=True,       # force le recyclage à 100% du plafond atteignable, sans coût
)


Presolve eliminates 0 constraints and 209006 variables.
Adjusted problem:
1364764 variables:
	6447 binary variables
	84 nonlinear variables
	1358233 linear variables
1402422 constraints; 5811451 nonzeros
	7 nonlinear constraints
	1402415 linear constraints
	1053361 equality constraints
	345570 inequality constraints
	3491 range constraints
2 objectives, all linear; 2 nonzeros.

Gurobi 13.0.2:   pre:dual = -1
  alg:method = 2
  bar:crossover = 0
  tech:threads = 0
  pre:passes = 3
  bar:convtol = 9.9999999999999995e-07
  pre:solve = -1
  iis:find = 1
Set parameter LogToConsole to value 1
  tech:outlev = 1

AMPL MP initial flat model has 1364764 variables (0 integer, 6447 binary);
Objectives: 1 linear; 
Constraints:  1402422 linear;

AMPL MP final model has 1368255 variables (0 integer, 6447 binary);
Objectives: 1 linear; 
Constraints:  1396960 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[x86] - Darwin 23.5.0 23F79)

CPU model: I

RuntimeError: Unexpected end of file while reading AMPL output.
Usually this is caused by the termination of the underlying AMPL interpreter.

In [8]:
run_pathway_materials(
    'recycling_materials_with_cost',
    verbose=True,
    materials_recycling=True,
    materials_recycling_cost=True,   # <- nouveau paramètre
    follow_objective=False,
)


Presolve eliminates 0 constraints and 209006 variables.
Adjusted problem:
1364764 variables:
	6447 binary variables
	84 nonlinear variables
	1358233 linear variables
1402422 constraints; 5877553 nonzeros
	7 nonlinear constraints
	1402415 linear constraints
	1053361 equality constraints
	345570 inequality constraints
	3491 range constraints
2 objectives, all linear; 2 nonzeros.

Gurobi 13.0.2:   pre:dual = -1
  alg:method = 2
  bar:crossover = 0
  tech:threads = 0
  pre:passes = 3
  bar:convtol = 9.9999999999999995e-07
  pre:solve = -1
  iis:find = 1
Set parameter LogToConsole to value 1
  tech:outlev = 1

AMPL MP initial flat model has 1364764 variables (0 integer, 6447 binary);
Objectives: 1 linear; 
Constraints:  1402422 linear;

AMPL MP final model has 1368255 variables (0 integer, 6447 binary);
Objectives: 1 linear; 
Constraints:  1396960 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[x86] - Darwin 23.5.0 23F79)

CPU model: I

{'TotalCost':               TotalCost
 YEAR_2020  73792.979153
 YEAR_2025  82743.904897
 YEAR_2030  73936.636256
 YEAR_2035  67772.641108
 YEAR_2040  64283.620296
 YEAR_2045  65525.687111
 YEAR_2050  70558.063452,
 'C_inv_phase':              C_inv_phase
 2015_2020  575020.385831
 2020_2025  100149.063478
 2025_2030   89246.339680
 2030_2035   65208.333485
 2035_2040   57036.580388
 2040_2045   52367.167906
 2045_2050   43041.994806,
 'C_inv_phase_tech':                                      C_inv_phase_tech
 Phases    Technologies                               
 2015_2020 AFC                                0.000000
           ALKALINE_ELECTROLYSIS              0.000000
           AL_MAKING                          0.000000
           AL_MAKING_HR                       0.000000
           AN_DIG                             0.000000
 ...                                               ...
 2045_2050 WIND_ONSHORE_DD_EESG               0.000000
           WIND_ONSHORE_DD_PMSG             851

In [9]:
run_pathway_materials(
    'recycling_materials_with_objective_cost',
    verbose=True,
    materials_recycling=True,
    materials_recycling_cost=True,   # <- nouveau paramètre
    follow_objective=True,
)


Presolve eliminates 0 constraints and 209006 variables.
Adjusted problem:
1364764 variables:
	6447 binary variables
	84 nonlinear variables
	1358233 linear variables
1402422 constraints; 6008645 nonzeros
	7 nonlinear constraints
	1402415 linear constraints
	1053361 equality constraints
	345570 inequality constraints
	3491 range constraints
2 objectives, all linear; 2 nonzeros.

Gurobi 13.0.2:   pre:dual = -1
  alg:method = 2
  bar:crossover = 0
  tech:threads = 0
  pre:passes = 3
  bar:convtol = 9.9999999999999995e-07
  pre:solve = -1
  iis:find = 1
Set parameter LogToConsole to value 1
  tech:outlev = 1

AMPL MP initial flat model has 1364764 variables (0 integer, 6447 binary);
Objectives: 1 linear; 
Constraints:  1402422 linear;

AMPL MP final model has 1368255 variables (0 integer, 6447 binary);
Objectives: 1 linear; 
Constraints:  1396960 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[x86] - Darwin 23.5.0 23F79)

CPU model: I

RuntimeError: Unexpected end of file while reading AMPL output.
Usually this is caused by the termination of the underlying AMPL interpreter.